# Per-viewer data columns demo

This notebook exercises `JdavizTableViewer.add_viewer_data_columns` through the UI.

It sets up a deconfigged Jdaviz app with:
- two images (`img_a`, `img_b`) and two spectra (`spec_a`, `spec_b`),
- a source catalog shown in a **Table** viewer,
- an **Image (demo)** viewer (seeded with `img_a`) and a **Spectrum (demo)** viewer (seeded with `spec_a`).

The Table viewer gets one read-only column **per viewer**, named `Data: <viewer>`. Each cell lists the data that belongs in that viewer for that row. When you click a row, every viewer is **cleared and repopulated** with exactly the data in its own column.

Run all cells, then follow the instructions near the bottom to click rows and watch the viewers update.

In [ ]:
import numpy as np
import astropy.units as u
from astropy.wcs import WCS
from astropy.table import Table
from astropy.nddata import NDData
from specutils import Spectrum

import jdaviz as jd

In [ ]:
viz = jd.gca()

## Build some synthetic data

Two images (sharing a simple TAN WCS), two 1D spectra, and a small source catalog (5 rows).

In [ ]:
wcs = WCS({'CTYPE1': 'RA---TAN', 'CUNIT1': 'deg', 'CDELT1': -0.0002777777778,
           'CRPIX1': 1, 'CRVAL1': 337.5202808,
           'CTYPE2': 'DEC--TAN', 'CUNIT2': 'deg', 'CDELT2': 0.0002777777778,
           'CRPIX2': 1, 'CRVAL2': -20.833333059999998})

arr = np.arange(10000).reshape((100, 100)).astype(float)
img_a_data = NDData(arr, wcs=wcs)
img_b_data = NDData(arr[::-1, ::-1], wcs=wcs)

rng = np.random.default_rng(42)
spectral_axis = np.linspace(4000, 7000, 200) * u.AA
spec_a = Spectrum(flux=(10 + rng.normal(0, 0.5, 200)) * u.Jy, spectral_axis=spectral_axis)
spec_b = Spectrum(flux=(5 + rng.normal(0, 0.5, 200)) * u.Jy, spectral_axis=spectral_axis)

catalog = Table()
catalog['ra'] = [337.50293, 337.52763, 337.52844, 337.47438, 337.47432] * u.deg
catalog['dec'] = [-20.81483, -20.80438, -20.82707, -20.82683, -20.80342] * u.deg
catalog['source_id'] = ['src_001', 'src_002', 'src_003', 'src_004', 'src_005']

## Load everything into the app

Loading the images and spectra auto-creates the default `Image` and `1D Spectrum` viewers. The catalog is loaded into a new `Table` viewer.

In [ ]:
viz.load(img_a_data, data_label='img_a')
viz.load(img_b_data, data_label='img_b')
viz.load(spec_a, format='1D Spectrum', data_label='spec_a')
viz.load(spec_b, format='1D Spectrum', data_label='spec_b')

# load the catalog into a Table viewer
ldr = viz.loaders['object']
ldr.object = catalog
ldr.format = 'Catalog'
ldr.importer.viewer.create_new = 'Table'
ldr.load()

## Create dedicated demo viewers

We resolve the real data-collection labels (loading an image applies a `[DATA]` suffix), then create an **Image (demo)** viewer seeded with `img_a` and a **Spectrum (demo)** viewer seeded with `spec_a`.

In [ ]:
dc_labels = list(viz._app.data_collection.labels)
img_a = next(lbl for lbl in dc_labels if lbl.startswith('img_a'))
img_b = next(lbl for lbl in dc_labels if lbl.startswith('img_b'))
spec_a_lbl = next(lbl for lbl in dc_labels if lbl.startswith('spec_a'))
spec_b_lbl = next(lbl for lbl in dc_labels if lbl.startswith('spec_b'))

vc = viz.new_viewers['Image']
vc.dataset = img_a
vc.viewer_label = 'Image (demo)'
vc()

vc = viz.new_viewers['1D Spectrum']
vc.dataset = spec_a_lbl
vc.viewer_label = 'Spectrum (demo)'
vc()

## Add one `Data: <viewer>` column per viewer

`add_viewer_data_columns` takes a mapping of viewer -> per-row data. Each viewer gets its own `Data: <viewer>` column. On row click, that viewer is cleared and shows exactly the data in its column for the clicked row.

| row | Data: Image (demo) | Data: Spectrum (demo) |
|-----|--------------------|-----------------------|
| src_001 | `img_b` | `spec_a` |
| src_002 | `img_a`, `img_b` | `spec_b` |
| src_003 | (none) | `spec_a`, `spec_b` |
| src_004 | `img_a` | (none) |
| src_005 | (none) | (none) |

In [ ]:
table_viewer = viz.viewers['Table']._obj.glue_viewer

viewer_data = {
    'Image (demo)': [
        [img_b],              # src_001
        [img_a, img_b],       # src_002
        [],                   # src_003
        [img_a],              # src_004
        [],                   # src_005
    ],
    'Spectrum (demo)': [
        [spec_a_lbl],             # src_001
        [spec_b_lbl],             # src_002
        [spec_a_lbl, spec_b_lbl], # src_003
        [],                       # src_004
        [],                       # src_005
    ],
}

table_viewer.add_viewer_data_columns(viewer_data)

## Try it in the UI

1. Show the app below.
2. In the **Table** viewer, click a row.
3. Watch both demo viewers get cleared and repopulated from their `Data: <viewer>` columns:
   - `src_001`: **Image (demo)** shows only `img_b`; **Spectrum (demo)** shows only `spec_a`.
   - `src_002`: **Image (demo)** shows both images; **Spectrum (demo)** swaps to `spec_b`.
   - `src_003`: **Image (demo)** is cleared; **Spectrum (demo)** shows both spectra.
   - `src_004`: **Image (demo)** shows only `img_a`; **Spectrum (demo)** is cleared.
   - `src_005`: both viewers are cleared.

Re-run the verification cell after clicking to confirm programmatically.

In [ ]:
viz.show()

## Verify (run after clicking rows)

Lists the datasets currently **visible** in each demo viewer.

In [ ]:
def visible(reference):
    viewer = viz._app.get_viewer(reference)
    return sorted(layer.layer.label for layer in viewer.layers if layer.visible)

print('Image (demo):   ', visible('Image (demo)'))
print('Spectrum (demo):', visible('Spectrum (demo)'))